In [ ]:
import glob
from pathlib import Path

dataset_location = '../../data/Flan/db_files/*.jsonl'
dataset_files = glob.glob(dataset_location)

dataset_files, len(dataset_files)

from datasets import load_dataset

data = []
dataset_names = []

for dataset_file in dataset_files:
    dataset = load_dataset("json", data_files = dataset_file, split='train')
    data.append(dataset)
    dataset_names.append(str(Path(dataset_file).name).replace("llama2_7b_lora-","").replace('.jsonl',''))

len(data)
dataset_names

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings


model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': False}
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs

)

In [ ]:
from tqdm import tqdm

sample_size = 300

all_embeddings = []
all_tokens = []

for train_data_set in tqdm(data):
    dataset_embedding = []
    strings = []
    for data_entry in train_data_set[:sample_size]['inputs']:
        strings.append(data_entry)
    dataset_embedding = embedding_model.embed_documents(strings)
    all_embeddings.append(dataset_embedding)

len(all_embeddings)

In [ ]:
import numpy as np
import torch
from MulticoreTSNE import MulticoreTSNE as TSNE
import pandas as pd

embeds = []

for emb in all_embeddings:
    embeds.append(np.array(emb))
all_embeds = np.concatenate(embeds)

len(all_embeds)

In [ ]:
from MulticoreTSNE import MulticoreTSNE as TSNE
import pandas as pd

tsne = TSNE(n_components=2,  verbose=True, n_jobs=80)

reduced_embeddings_tsne = tsne.fit_transform(all_embeds)

len(reduced_embeddings_tsne)

In [ ]:
labels = []
for datset_name in dataset_names:
    labels  = labels + [datset_name] * sample_size

In [ ]:
import plotly.express as px

df = pd.DataFrame(reduced_embeddings_tsne, columns = ['x', 'y'])
df['labels'] = labels

fig = px.scatter(df, x='x', y='y', color = 'labels', opacity=0.4, title = 'Reduction with tSNE')
fig.show()

In [ ]:
import matplotlib.pyplot as plt

pd.options.mode.chained_assignment = None

df = pd.DataFrame(reduced_embeddings_tsne, columns = ['x', 'y'])
df['labels'] = labels
sample_labels = df['labels'].unique()
cmap = plt.colormaps.get_cmap('hsv').resampled(len(sample_labels))

fig, ax = plt.subplots(layout='tight')

for idx_sample_value, sample_value in enumerate(sample_labels):
    sub_data = df.loc[df['labels'] == sample_value]
    ax.scatter(x=sub_data['x'], y=sub_data['y'], s=50, c=[cmap(idx_sample_value)],
               alpha=0.5, marker = 'o', edgecolor='black', linewidth=0.5,
               label=f"{sample_value}")
    sub_data.drop('labels', axis=1, inplace=True)
    npr = sub_data.to_numpy()


ax.legend(sample_labels, ncol = 5, shadow = True, bbox_to_anchor=(1, -0.1))
fig.set_size_inches(12, 6)
plt.show()
fig.savefig("../figs/flanv2_embed.pdf",bbox_inches = 'tight',
    pad_inches = 0)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

all_embeddings = []
all_tokens = []

for train_data_set in tqdm(data):
    dataset_embedding = []
    strings = []
    for data_entry in train_data_set[:800]['inputs']:
        strings.append(data_entry)
    dataset_embedding = embedding_model.embed_documents(strings)
    all_embeddings.append(dataset_embedding)
len(all_embeddings)

embeds = [np.array(emb) for emb in all_embeddings]

centroids = [np.asarray(e).mean(axis=0) for e in embeds]

pairwise_css = cosine_similarity(centroids)


In [ ]:
import seaborn as sns
import matplotlib.pylab as plt


num_ticks = len(dataset_names)
yticks = np.linspace(0, len(dataset_names) - 1, num_ticks, dtype=np.int8)
yticklabels = [dataset_names[idx] for idx in yticks]

sns.set_theme(rc={'figure.figsize':(11.7,8.27)})
ax = sns.heatmap(pairwise_css, linewidth=0.5, xticklabels=yticklabels, yticklabels=yticklabels)

plt.show()
ax.figure.savefig("../figs/flanv2_cosine.pdf",bbox_inches = 'tight',
    pad_inches = 0)